# Fill-scenario diagnostics — weather FTC TP strategy

Visualises Session 2.6 output for canonical `(p_in=0.60, p_out=0.90, policy='all')`. All compute uses functions in `data_infra.weather_analysis`. Re-running cell 2 takes ~20s (one trades-parquet scan). Window comparison + any-fill cells each take ~10-40s.

In [ ]:
import sys
import importlib
from pathlib import Path
RESEARCH = Path.cwd().resolve()
while RESEARCH.name != 'research' and RESEARCH.parent != RESEARCH:
    RESEARCH = RESEARCH.parent
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from data_infra import weather_analysis as wa
importlib.reload(wa)  # pick up new functions on re-run without restarting kernel

inst = pd.read_parquet(wa.DEFAULT_DATA_DIR / 'weather_tail_per_instance.parquet')
p_wide = wa.pivot_inst_to_wide(inst)
P_IN, P_OUT = 0.60, 0.90
print(f'p_wide: {len(p_wide):,} rows   canonical: p_in={P_IN}, p_out={P_OUT}')

In [ ]:
# ~20s on first run; one ASOF-style scan against the trades parquet.
out, audit = wa.eval_pair(p_wide, P_IN, P_OUT, return_audit=True)
scen = wa.compute_fill_scenarios(audit)
diag = wa.slippage_diagnostic(scen)
print(f'n_entries: {out["n_entries"]}   any_leg_high_fallback: {diag["any_leg_high_fallback"]}')
print(f'edges: next_same_dir={scen["pnl_next_same_dir"].mean():+.4f}  '
      f'midpoint={scen["pnl_midpoint"].mean():+.4f}  '
      f'next_opp_dir={scen["pnl_next_opp_dir"].mean():+.4f}')

In [ ]:
# Chart 1 — edge by scenario
edges = {'next_same_dir': scen['pnl_next_same_dir'].mean(),
         'midpoint':      scen['pnl_midpoint'].mean(),
         'next_opp_dir':  scen['pnl_next_opp_dir'].mean()}
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(edges.keys(), edges.values(),
       color=['#c8313a' if v < 0 else '#3a7a3a' for v in edges.values()])
ax.axhline(0, color='k', lw=0.5)
for i, (k, v) in enumerate(edges.items()):
    ax.text(i, v + 0.001*np.sign(v) - (0.002 if v<0 else 0),
            f'{v:+.4f}\n({100*v/P_IN:+.2f}%)',
            ha='center', va='bottom' if v >= 0 else 'top')
ax.set_ylabel('edge ($/share)')
ax.set_title(f'Edge per fill scenario  (p_in={P_IN}, p_out={P_OUT}, n={len(scen)})')
plt.tight_layout(); plt.show()

In [ ]:
# Chart 2 — fallback rate per leg × direction
legs = pd.DataFrame(diag['legs'])
labels = [f'{r["leg"]}/{r["direction"]}' for _, r in legs.iterrows()]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(labels, legs['fallback_pct'],
       color=['#c8313a' if v > 0.5 else '#3a7a3a' for v in legs['fallback_pct']])
ax.axhline(0.5, color='k', ls='--', lw=0.8, label='50% threshold')
ax.set_ylim(0, 1); ax.set_ylabel('fallback share')
ax.set_title('Fallback rate per leg  (red = constant_slippage_with_labelling)')
for i, r in legs.iterrows():
    ax.text(i, r['fallback_pct'] + 0.02, f'{r["fallback_pct"]:.2f}', ha='center')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Chart 3 — slippage distribution per leg (real next-fills only; clipped to [-15, 15]c)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), sharey=True)
for ax, leg_row in zip(axes, diag['legs']):
    leg, dirn = leg_row['leg'], leg_row['direction']
    trig = P_IN if leg == 'entry' else P_OUT
    price_col, src_col = f'{leg}_next_{dirn}_price', f'{leg}_next_{dirn}_source'
    sub = scen if leg == 'entry' else scen[scen['bucket'].eq('tp')]
    nf = sub[sub[src_col] == 'next_fill']
    if len(nf):
        slip = ((nf[price_col].astype(float) - trig) * 100).clip(-15, 15)
        ax.hist(slip, bins=30, color='steelblue', alpha=0.85)
        ax.axvline(slip.median(), color='black', lw=1, label=f'med={slip.median():+.1f}¢')
        ax.legend(fontsize=8)
    ax.axvline(0, color='gray', lw=0.5)
    ax.set_title(f'{leg}/{dirn}  fb={leg_row["fallback_pct"]:.2f}')
    ax.set_xlabel('slip (cents)')
axes[0].set_ylabel('count')
fig.suptitle('Slippage distribution (real next-fills only; fallback rows excluded)', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Chart 4 — spread estimate per leg (entry: ASK-BID; exit: BID-ASK)
e_spread = (scen['spread_estimate_entry'] * 100).clip(-15, 15)
x_spread = (scen.loc[scen['bucket'].eq('tp'), 'spread_estimate_exit'] * 100).clip(-15, 15)
spread_st = {s['leg']: s for s in diag['spread']}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, ser, label, st in [(a1, e_spread, 'entry', spread_st['entry']),
                            (a2, x_spread, 'exit',  spread_st['exit'])]:
    ax.hist(ser, bins=30, color='steelblue', alpha=0.85)
    ax.axvline(0, color='black', lw=0.8)
    ax.axvline(ser.median(), color='red', lw=1, ls='--', label=f'med={ser.median():+.1f}¢')
    ax.set_xlabel('next_same_dir − next_opp_dir (cents)')
    ax.set_title(f'spread_estimate {label}   crossed_share={st["crossed_market_share"]:.1%}')
    ax.legend(fontsize=9)
a1.set_ylabel('count'); plt.tight_layout(); plt.show()

In [ ]:
# Chart 5 — raw market activity post-cross (any side, any direction, 30-min cap)
any_fill = wa.time_to_first_any_fill_diagnostic(p_wide, p_in=P_IN, max_seconds=1800)
ts = [30, 60, 120, 300, 600, 1800]
shares = [any_fill[f'pct_within_{t}s'] for t in ts[:-1]] + [1 - any_fill['pct_no_fill_within_max']]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(ts, shares, marker='o', color='steelblue', lw=2)
ax.axvline(300, color='gray',  ls='--', lw=0.8, label='5-min window')
ax.axvline(600, color='red',   ls='--', lw=0.8, label='10-min window')
ax.set_xscale('log'); ax.set_ylim(0, 1)
ax.set_xlabel('time after cross (s, log)')
ax.set_ylabel('cumulative share with any next fill')
ax.set_title(f'Time to first OTHER fill in same market  '
             f'(n={any_fill["n_anchors"]}; lag p50={any_fill["lag_median_s"]:.0f}s; '
             f'{any_fill["pct_no_fill_within_max"]:.1%} never)')
for t, s in zip(ts, shares):
    ax.text(t, s + 0.025, f'{s:.1%}', ha='center', fontsize=8)
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Chart 6 — window comparison (5min vs 10min)  ~40s
win = wa.compare_windows_diagnostic(p_wide, P_IN, P_OUT, windows_seconds=(300, 600))
ws = sorted(win['windows'].keys())
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5))
scenarios = ['next_same_dir', 'midpoint', 'next_opp_dir']
edge_data = {f'{w}s': [win['windows'][w][f'edge_{s}'] for s in scenarios] for w in ws}
pd.DataFrame(edge_data, index=scenarios).plot.bar(ax=a1, color=['#c8a04c', '#4c7ac8'])
a1.axhline(0, color='k', lw=0.5)
a1.set_title('Edge by scenario × window'); a1.set_ylabel('edge ($/share)')
a1.set_xticklabels(scenarios, rotation=0)
fb_legs = ['entry_same_dir', 'entry_opp_dir', 'exit_same_dir', 'exit_opp_dir']
fb_data = {f'{w}s': [win['windows'][w][f'fallback_rate_{l}'] for l in fb_legs] for w in ws}
pd.DataFrame(fb_data, index=fb_legs).plot.bar(ax=a2, color=['#c8a04c', '#4c7ac8'])
a2.axhline(0.5, color='k', ls='--', lw=0.8)
a2.set_title('Fallback rate by leg × window'); a2.set_ylim(0, 1)
a2.set_xticklabels(fb_legs, rotation=20, ha='right')
plt.tight_layout(); plt.show()
d = win['delta_short_to_long']
print(f'Δ {d["short_seconds"]}s → {d["long_seconds"]}s:  '
      f'fallback drop entry_same_dir={d["fallback_drop_entry_same_dir_pp"]:+.2f}pp, '
      f'edge Δ same_dir={d["edge_change_same_dir_cents"]:+.2f}¢, '
      f'midpoint={d["edge_change_midpoint_cents"]:+.2f}¢, '
      f'opp_dir={d["edge_change_opp_dir_cents"]:+.2f}¢')

## Proposal B — data-supported subsets

Filter to entries where real next-fills exist (not fallback). Tests whether 
"only trade when the market is liquid" recovers the edge.

In [ ]:
# Chart 7 — subset summary at canonical (no extra parquet scans; reuses `scen`)
baseline_mask = pd.Series(True, index=scen.index)
taker_mask = scen['entry_next_same_dir_source'] == 'next_fill'
maker_mask = scen['entry_next_opp_dir_source'] == 'next_fill'
inter_mask = taker_mask & maker_mask
summary = pd.DataFrame([
    wa.subset_pnl_summary(scen, baseline_mask, 'baseline',     p_in=P_IN),
    wa.subset_pnl_summary(scen, taker_mask,    'taker_filt',   p_in=P_IN),
    wa.subset_pnl_summary(scen, maker_mask,    'maker_filt',   p_in=P_IN),
    wa.subset_pnl_summary(scen, inter_mask,    'intersection', p_in=P_IN),
]).set_index('label')
print(summary[['n','p_tp','edge_next_same_dir','edge_midpoint','edge_next_opp_dir',
               'mean_entry_slip_cents']].round({
    'p_tp':3,'edge_next_same_dir':4,'edge_midpoint':4,'edge_next_opp_dir':4,
    'mean_entry_slip_cents':2}))
fig, ax = plt.subplots(figsize=(8, 3.5))
summary[['edge_next_same_dir','edge_midpoint','edge_next_opp_dir']].plot.bar(
    ax=ax, color=['#c8313a','#888','#3a7a3a'])
ax.axhline(0, color='k', lw=0.5)
ax.set_title('Edge per scenario × subset (canonical p_in=0.60, p_out=0.90)')
ax.set_ylabel('edge ($/share)'); ax.set_xticklabels(summary.index, rotation=0)
ax.legend(['next_same_dir','midpoint','next_opp_dir'], loc='lower left')
plt.tight_layout(); plt.show()

In [ ]:
# Chart 8 — selection bias: top cities baseline vs maker_filt subset
bias = wa.subset_selection_bias(scen, p_wide, maker_mask, baseline_mask=baseline_mask)
base_share = {c['city']: c['share'] for c in bias['top_cities_baseline']}
sub_share  = {c['city']: c['share'] for c in bias['top_cities_subset']}
cities = list(dict.fromkeys(list(base_share.keys()) + list(sub_share.keys())))
df_b = pd.DataFrame({'baseline':[base_share.get(c,0) for c in cities],
                     'maker_filt':[sub_share.get(c,0) for c in cities]}, index=cities)
df_b.index = [c.replace('highest-temperature-in-','').replace('lowest-temperature-in-','')
              for c in df_b.index]
fig, ax = plt.subplots(figsize=(10, 3.5))
df_b.plot.bar(ax=ax, color=['#888','#3a7a3a'])
ax.set_title(f'Top cities: baseline vs maker_filt subset '
             f'(n_sub={bias["n_subset"]} = {bias["subset_share_of_baseline"]:.1%} of baseline)')
ax.set_ylabel('share within set'); ax.set_xticklabels(df_b.index, rotation=20, ha='right')
plt.tight_layout(); plt.show()
bh = bias['hours_to_resolution_baseline']; sh = bias['hours_to_resolution_subset']
print(f'hours-to-resolution at entry (p10/p50/p90):')
print(f'  baseline  : {bh["p10"]:.1f} / {bh["median"]:.1f} / {bh["p90"]:.1f}')
print(f'  maker_filt: {sh["p10"]:.1f} / {sh["median"]:.1f} / {sh["p90"]:.1f}')

## WS-passive execution model

If the bot detects via CLOB WebSocket and posts passive limit orders at the touch, 
the next-fill *price* proxies don't apply — fills happen at exactly the posted price 
(p_in for entry, p_out for exit on TP). What matters instead is *fill rate*:

- entry fills when an aggressive seller arrives (≈ `entry_next_opp_dir_source == 'next_fill'`)
- exit fills at p_out under one of two assumptions:
  - **optimistic**: if price reached p_out, we were the best ask and got lifted
  - **strict**: also require a real aggressive-buy print in the exit window

In [ ]:
# Chart 9 — WS-passive headline vs proxy scenarios (reuses `scen`, no parquet scan)
_aug_opt, sum_opt = wa.passive_pnl_from_audit(scen, exit_passive='optimistic')
_aug_str, sum_str = wa.passive_pnl_from_audit(scen, exit_passive='strict')
rows = [
    {'model':'proxy taker (cross spread)',          'n_trades':len(scen),
     'edge_per_trade':float(scen['pnl_next_same_dir'].mean()),
     'roi_pct':100*float(scen['pnl_next_same_dir'].mean())/P_IN},
    {'model':'proxy maker_best (fallback-heavy)',  'n_trades':len(scen),
     'edge_per_trade':float(scen['pnl_next_opp_dir'].mean()),
     'roi_pct':100*float(scen['pnl_next_opp_dir'].mean())/P_IN},
    {'model':'WS-passive optimistic exit',          'n_trades':sum_opt['n_entry_filled'],
     'edge_per_trade':sum_opt['edge_per_filled'], 'roi_pct':sum_opt['roi_per_filled_pct']},
    {'model':'WS-passive strict exit',              'n_trades':sum_str['n_entry_filled'],
     'edge_per_trade':sum_str['edge_per_filled'], 'roi_pct':sum_str['roi_per_filled_pct']},
]
df = pd.DataFrame(rows).set_index('model')
print(df.round({'edge_per_trade':4,'roi_pct':2}))
print(f"\nfill_rate (WS-passive entry): {sum_opt['fill_rate']:.1%}  ({sum_opt['n_entry_filled']} of {sum_opt['n_total']})")
print(f"under optimistic exit: tp={sum_opt['p_tp_of_filled']:.3f}  hold_win={sum_opt['p_hold_win_of_filled']:.3f}  hold_chop={sum_opt['p_hold_chop_of_filled']:.3f}")
print(f"under strict     exit: tp={sum_str['p_tp_of_filled']:.3f}  hold_win={sum_str['p_hold_win_of_filled']:.3f}  hold_chop={sum_str['p_hold_chop_of_filled']:.3f}")
fig, ax = plt.subplots(figsize=(9, 3.5))
colors = ['#c8a04c' if v < 0 else '#3a7a3a' for v in df['edge_per_trade']]
ax.bar(df.index, df['edge_per_trade'], color=colors)
ax.axhline(0, color='k', lw=0.5)
ax.set_ylabel('edge per trade ($/share)')
ax.set_title('Headline edge per trade by execution model  (canonical p_in=0.60, p_out=0.90)')
for i, (k, v) in enumerate(df['edge_per_trade'].items()):
    ax.text(i, v + 0.001*np.sign(v), f"{v:+.4f}\n(ROI {df.loc[k,'roi_pct']:+.2f}%)\nn={df.loc[k,'n_trades']}",
            ha='center', va='bottom' if v>=0 else 'top', fontsize=8)
plt.xticks(rotation=15, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# Chart 10 — top 5 grid cells under WS-passive (optimistic exit), if grid was saved
grid_path = wa.DEFAULT_DATA_DIR / 'passive_grid_canonical.parquet'
if grid_path.exists():
    grid = pd.read_parquet(grid_path)
    top = grid.sort_values('edge_per_filled', ascending=False).head(5).set_index(['p_in','p_out'])
    print(top[['n_entry_filled','fill_rate','p_tp_of_filled','edge_per_filled','roi_per_filled_pct','total_pnl_per_entered']]
          .round({'fill_rate':3,'p_tp_of_filled':3,'edge_per_filled':4,'roi_per_filled_pct':2,'total_pnl_per_entered':1}))
    fig, ax = plt.subplots(figsize=(9, 3.5))
    pivot = grid.pivot_table(index='p_in', columns='p_out', values='edge_per_filled')
    im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=-0.02, vmax=0.03)
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index)
    ax.set_xlabel('p_out'); ax.set_ylabel('p_in')
    ax.set_title('WS-passive edge_per_filled by (p_in, p_out)')
    for i, p_in_v in enumerate(pivot.index):
        for j, p_out_v in enumerate(pivot.columns):
            v = pivot.iloc[i,j]
            if not pd.isna(v):
                ax.text(j, i, f'{v:+.3f}', ha='center', va='center', fontsize=8,
                        color='black' if abs(v) < 0.015 else 'white')
    plt.colorbar(im, ax=ax, label='edge $/share')
    plt.tight_layout(); plt.show()
else:
    print(f'grid not found at {grid_path}. Run scripts/passive_analysis.py first.')

**Caveat.** These are PROXIES for execution cost from realised fills, not literal 
bid/ask. The gap between scenarios is a sensitivity range, not a precise estimate. 
Where a leg's fallback rate > 50% (red bars in chart 2), the scenario edge is 
dominated by the `fallback_cents` parameter — read those rows as *constant 
slippage with extra labelling*.

**Proposal B finding.** Filtering to data-supported subsets does NOT improve edge 
— it makes it WORSE because the active markets are exactly the most-costly to 
execute under the proxy model.

**WS-passive finding.** With WebSocket detection + passive limit posting, the 
next-fill price proxies don't apply — fills happen at exactly the posted price. 
Under the optimistic exit assumption, the canonical cell flips to +0.87% ROI on 
26% fill rate (2,330 fills/yr); the best grid cell is (p_in=0.50, p_out=0.90) at 
**+6.0% ROI on 2,283 fills/yr**. Under the strict exit assumption (no queue priority 
at exit), even the canonical reverts to −2.5% ROI. **Real performance depends on 
your actual queue position** — try small-scale first to measure where between 
optimistic and strict you actually sit. See `notes/weather_ftc_state.md` for the 
revised verdict.